# 04 Embedding Generation & Multi-Vector Representations

**Real-World Scenario**: E-Commerce Product Catalog Multi-Vector Search.

This notebook demonstrates **Matryoshka Representation Learning (MRL)** dimension slicing ($1536	ext{d} ightarrow 256	ext{d}$) to reduce RAM consumption by $83.3\%$ while maintaining vector search quality.

## Step 1: Environment Setup

In [ ]:
import os
from dotenv import load_dotenv

# Load API keys from repository root .env
load_dotenv()

# Create hidden vector database directory for local index persistence
os.makedirs(".vectordb", exist_ok=True)

print("=== Repository Environment Setup ===")
print("OPENAI_API_KEY present:", bool(os.getenv("OPENAI_API_KEY")))
print("GEMINI_API_KEY present:", bool(os.getenv("GEMINI_API_KEY")))
print("Hidden DB Directory Path:", os.path.abspath(".vectordb"))


## Step 2: Matryoshka Representation Learning (MRL) Dimension Slicing

In [ ]:
from langchain_openai import OpenAIEmbeddings

if os.getenv("OPENAI_API_KEY"):
    # 1. Full 1536d Embedding
    embed_full = OpenAIEmbeddings(model="text-embedding-3-small")
    vec_full = embed_full.embed_query("Enterprise Ergonomic Standing Desk Dual Motor")
    
    # 2. MRL Sliced 256d Embedding
    embed_256 = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=256)
    vec_256 = embed_256.embed_query("Enterprise Ergonomic Standing Desk Dual Motor")
    
    print("=== Matryoshka Embedding Slicing (MRL) Benchmark ===")
    print(f"Full Vector Dimension:   {len(vec_full)}")
    print(f"Sliced Vector Dimension: {len(vec_256)}")
    print(f"RAM Memory Reduction Factor: {len(vec_full) / len(vec_256):.1f}x Savings!")


## Step 3: Indexing MRL Vectors into Hidden `.vectordb/` Directory

In [ ]:
from langchain_community.vectorstores import FAISS

products = [
    "Product A: Ergonomic Standing Desk Dual Motor Height Adjustable 48x30 inch.",
    "Product B: Noise Cancelling Wireless Headphones 40-hour Battery Life.",
    "Product C: UltraWide Curved Gaming Monitor 34-inch 144Hz 1ms response."
]

if os.getenv("OPENAI_API_KEY"):
    embed_256 = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=256)
    vectorstore = FAISS.from_texts(products, embed_256)
    save_path = os.path.join(".vectordb", "04_mrl_256_index")
    vectorstore.save_local(save_path)
    
    results = vectorstore.similarity_search("standing desk adjustable", k=1)
    print("=== MRL 256d Vector Search Result ===")
    print(results[0].page_content)
    print(f"Saved MRL index to hidden folder: {save_path}")
